# VQ-VAE MNIST

In this notebook we train a **Vector Quantized Variational Auto-Encoder (VQ-VAE)**, proposed by [van den Oord et al. 2017](https://arxiv.org/abs/1711.00937).

The VQ-VAE differs from a standard [VAE](vae_mnist.ipynb) in one essential way: instead of a *continuous* Gaussian latent, the encoder output is snapped to the nearest entry of a learned **codebook**. The latent representation of an image is therefore a small grid of **discrete integer indices** rather than a vector of reals.

That change has a consequence that shapes this whole notebook. A VAE can generate by sampling its latent from the prior $\mathcal{N}(0, I)$, because it *has* a prior by construction. A VQ-VAE has no such prior over its index grid — so a trained VQ-VAE **cannot generate at all on its own**, only reconstruct. Generation requires a *second* model, trained afterwards, that learns the distribution over index grids.

This gives the two-stage structure the paper describes, and the two classes in `lightning_uq_box`:

| Stage | Class | Trains | Gives you |
| --- | --- | --- | --- |
| 1 | `VQVAE` | encoder + codebook + decoder | reconstruction, a discrete index grid |
| 2 | `VQVAEPrior` | a `PixelCNN` over the index grid (VQ-VAE frozen) | **generation**, by ancestral sampling |

We will do both, and finish by sampling novel digits from the prior.

In [ ]:
%%capture
%pip install git+https://github.com/lightning-uq-box/lightning-uq-box.git

## Imports

In [ ]:
import os
import tempfile
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from lightning import LightningDataModule, Trainer
from lightning.pytorch import seed_everything
from lightning.pytorch.loggers import CSVLogger
from segmentation_models_pytorch.encoders import TimmUniversalEncoder
from torch import Tensor
from torchvision.utils import make_grid

from lightning_uq_box.uq_methods import VQVAE, VQVAELoss, VQVAEPrior
from lightning_uq_box.viz_utils import plot_training_metrics

plt.rcParams["figure.figsize"] = [14, 5]

%load_ext autoreload
%autoreload 2

In [ ]:
seed_everything(0)

In [ ]:
my_temp_dir = tempfile.mkdtemp()

## Datamodule

As with the VAE, the target *is* the input, since the task is reconstruction. Note there is no augmentation here — random crops and flips are classification tricks and would only make the reconstruction target noisier.

In [ ]:
def collate_fn(batch):
    """Collate function for dataloader as dictionary."""
    images, targets = zip(*batch)
    images = torch.stack(images)
    targets = torch.tensor(targets)
    # the target is also the image, since we want to reconstruct it
    return {"input": images, "target": images, "labels": targets}


class MNISTDatamodule(LightningDataModule):
    def __init__(
        self, root: str, batch_size: int = 64, num_workers: int = 0, resize: int = 32
    ):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.root = root
        self.resize = resize

    def _transform(self):
        return torchvision.transforms.Compose(
            [
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Resize((self.resize, self.resize)),
                # normalize to [-1, 1], the range the decoder outputs
                torchvision.transforms.Normalize((0.5,), (0.5,)),
            ]
        )

    def setup(self, stage: str) -> None:
        """Set up the datasets."""
        if stage in ["fit", "validate"]:
            mnist_train = torchvision.datasets.MNIST(
                self.root, train=True, download=True, transform=self._transform()
            )
            self.mnist_train, self.mnist_val = torch.utils.data.random_split(
                mnist_train, [55000, 5000]
            )
        if stage in ["test"]:
            self.mnist_test = torchvision.datasets.MNIST(
                self.root, train=False, download=True, transform=self._transform()
            )

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.mnist_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            collate_fn=collate_fn,
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.mnist_val,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            collate_fn=collate_fn,
        )

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.mnist_test,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            collate_fn=collate_fn,
        )

## Stage 1: the VQ-VAE

### Geometry

The size of the latent index grid is decided entirely by the encoder, and it is worth being deliberate about it, because it sets the cost of stage 2.

The grid is `img_size // encoder.output_stride`. A `resnet18` at `depth=2` has `output_stride=4`, so a 32x32 input gives an **8x8** grid — 64 discrete indices per image. That is the geometry the paper uses for CIFAR-10.

Stage 2 will need one sequential forward pass *per grid position*, so an 8x8 grid costs 64 passes per sample. Doubling the grid to 16x16 would quadruple that to 256. Let's confirm what we get before training anything.

In [ ]:
encoder = TimmUniversalEncoder("resnet18", depth=2, in_channels=1)
feature = encoder(torch.randn(1, 1, 32, 32))[-1]
print(f"encoder output_stride: {encoder.output_stride}")
print(f"last feature map:      {tuple(feature.shape)}")
print(
    f"latent grid:           {32 // encoder.output_stride}x{32 // encoder.output_stride}"
)

### The model

Two arguments are specific to vector quantization:

- **`codebook_size`** — how many discrete codes are available (512 in the paper). This is the vocabulary size stage 2 will model.
- **`sample_codebook_temp`** — temperature for stochastic codebook sampling. This is what makes `predict_step` produce *uncertainty*: with `num_samples > 1` each forward pass draws a slightly different code assignment, and the spread across draws is the returned `pred_uct`. Leave it at `0.0` and every draw is identical, so the uncertainty would be exactly zero.

`VQVAELoss` has one parameter, the commitment cost $\beta$, which weights how strongly the encoder is pushed to commit to the code it selected. The paper uses 0.25 and reports results are robust between 0.1 and 2.0.

In [ ]:
vqvae = VQVAE(
    encoder=TimmUniversalEncoder("resnet18", depth=2, in_channels=1),
    out_channels=1,  # MNIST is single channel
    img_size=32,
    codebook_size=512,
    sample_codebook_temp=1.0,  # > 0 so that predict_step gives non-zero uncertainty
    num_samples=5,
    loss_fn=VQVAELoss(commit_scale=0.25),  # beta = 0.25, the paper's value
    optimizer=partial(torch.optim.Adam, lr=3e-4),
)

Let's confirm the latent really is a grid of integers, since this is the whole point of the method and the easiest thing to get wrong.

In [ ]:
vqvae.configure_model()
with torch.no_grad():
    quantized, indices, commit_loss = vqvae.encode_img_to_latent(
        torch.randn(4, 1, 32, 32)
    )

print(f"quantized: {tuple(quantized.shape)}  {quantized.dtype}")
print(f"indices:   {tuple(indices.shape)}  {indices.dtype}")
print(
    f"index range: [{indices.min().item()}, {indices.max().item()}] out of {vqvae.codebook_size} codes"
)

`indices` is `(B, 8, 8)` and of dtype `int64` — a genuine spatial grid of integers, which is exactly what a PixelCNN can be trained on in stage 2.

In [ ]:
logger = CSVLogger(os.path.join(my_temp_dir, "stage1"))
trainer = Trainer(
    max_epochs=15,
    accelerator="gpu",
    devices=[0],
    logger=logger,
    enable_checkpointing=False,
    log_every_n_steps=20,
)

In [ ]:
datamodule = MNISTDatamodule(root="./data", batch_size=128, num_workers=2)
trainer.fit(vqvae, datamodule)

### Training metrics

Alongside the reconstruction loss, `VQVAE` logs two **codebook diagnostics**, and these matter more than the loss:

- **`train_perplexity`** — the effective number of codes in use. Its ceiling is `codebook_size`.
- **`train_codebook_usage`** — the fraction of the codebook that received any assignment.

The characteristic VQ-VAE failure is **codebook collapse**: the model routes everything through a handful of codes, the reconstruction loss still looks reasonable, and stage 2 then has almost nothing to model. A low loss with collapsed usage is a failure, not a success, so always read these two next to the loss.

In [ ]:
fig = plot_training_metrics(
    os.path.join(my_temp_dir, "stage1", "lightning_logs"),
    ["train_loss", "train_perplexity", "train_codebook_usage"],
)

### Reconstruction with uncertainty

`predict_step` returns `pred` (the mean over `num_samples` draws) and `pred_uct` (their standard deviation). The uncertainty comes from stochastic codebook sampling, so it tends to be highest where the image is ambiguous about *which* code applies — typically stroke edges.

In [ ]:
def plot_vqvae_reconstruction(batch: dict[str, Tensor], pred_dict: dict[str, Tensor]):
    """Plot VQ-VAE reconstruction results.

    Args:
        batch: input batch from the dataloader
        pred_dict: prediction dictionary from ``VQVAE.predict_step``
    """
    _fig, axs = plt.subplots(4, 3, figsize=(12, 16))
    for i in range(4):
        rand_idx = np.random.randint(0, batch["input"].shape[0])
        axs[i, 0].imshow(batch["input"][rand_idx, 0], cmap="gray")
        axs[i, 1].imshow(pred_dict["pred"][rand_idx, 0], cmap="gray")
        im = axs[i, 2].imshow(pred_dict["pred_uct"][rand_idx, 0], cmap="viridis")
        plt.colorbar(im, ax=axs[i, 2], fraction=0.046)
        for j in range(3):
            axs[i, j].axis("off")

    axs[0, 0].set_title("Input", fontsize=18)
    axs[0, 1].set_title("Reconstruction", fontsize=18)
    axs[0, 2].set_title("Uncertainty", fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
datamodule.setup("test")
with torch.no_grad():
    batch = next(iter(datamodule.test_dataloader()))
    pred_dict = vqvae.predict_step(batch["input"])

print(f"mean predictive uncertainty: {pred_dict['pred_uct'].mean():.5f}")
plot_vqvae_reconstruction(batch, pred_dict)

### Why we cannot generate yet

It is worth making the limitation concrete rather than just asserting it. Calling `sample()` on a stage-1 VQ-VAE raises, because there is no distribution to sample indices from:

In [ ]:
try:
    vqvae.sample(num_samples=4)
except NotImplementedError as e:
    print(f"NotImplementedError: {e}")

## Stage 2: a PixelCNN prior over the index grid

Now we learn $p(\text{index grid})$ with a **gated PixelCNN**, which factorizes the joint distribution over the 8x8 grid autoregressively in raster-scan order:

$$p(z_{1:64}) = \prod_{i} p(z_i \mid z_{<i})$$

Causality — each position seeing only positions before it — is enforced by masked convolutions, using the vertical/horizontal stack design from [van den Oord et al. 2016](https://arxiv.org/abs/1606.05328) that avoids the blind spot a naive mask creates.

`VQVAEPrior` takes the trained `VQVAE` and handles the bookkeeping that is easy to get wrong:

- the VQ-VAE is **frozen and set to eval**, so stage 2 cannot disturb stage 1;
- only `pixel_cnn.parameters()` are handed to the optimizer;
- if you pass `vq_vae_ckpt_path`, the checkpoint is loaded **strictly**, so an architecture mismatch fails loudly instead of quietly training a prior against random weights.

Here we pass the in-memory model we just trained, so no checkpoint path is needed.

In [ ]:
prior = VQVAEPrior(
    vq_vae=vqvae, c_hidden=64, optimizer=partial(torch.optim.Adam, lr=3e-4)
)

n_trainable = sum(p.numel() for p in prior.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in prior.parameters() if not p.requires_grad)
print(f"trainable (PixelCNN): {n_trainable:,}")
print(f"frozen (VQ-VAE):      {n_frozen:,}")

The training objective is plain cross-entropy over the `codebook_size` classes at each of the 64 grid positions. A useful reference point: an untrained prior has loss $\ln(512) = 6.24$ and accuracy $1/512 \approx 0.2\%$, so anything meaningfully below and above those respectively means the prior is learning real structure.

In [ ]:
print(f"untrained-prior reference loss: ln(512) = {np.log(512):.3f}")

logger_prior = CSVLogger(os.path.join(my_temp_dir, "stage2"))
trainer_prior = Trainer(
    max_epochs=15,
    accelerator="gpu",
    devices=[0],
    logger=logger_prior,
    enable_checkpointing=False,
    log_every_n_steps=20,
)
trainer_prior.fit(prior, datamodule)

In [ ]:
fig = plot_training_metrics(
    os.path.join(my_temp_dir, "stage2", "lightning_logs"), ["train_loss", "train_acc"]
)

### Generating samples by ancestral sampling

This is the payoff. To draw a novel image we sample the index grid one position at a time in raster-scan order: predict the distribution at position $i$ given everything already sampled, draw from it, **write it back**, and move on. Only once the full 8x8 grid exists do we look up the corresponding codebook vectors and run the decoder once.

The write-back is the essential part — it is what makes the samples globally coherent rather than 64 independent draws. The cost is that the grid positions cannot be parallelized: an 8x8 grid means 64 sequential forward passes per batch (the batch itself is parallel).

`temperature` controls the sharpness of each draw: below 1.0 gives more typical but less diverse samples, above 1.0 the reverse.

In [ ]:
with torch.no_grad():
    samples = prior.sample(num_samples=16, temperature=1.0)

print(f"samples: {tuple(samples.shape)}")

grid = make_grid(samples.cpu(), nrow=4, normalize=True, padding=2)
fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title("Ancestral samples from the PixelCNN prior", fontsize=16)
ax.axis("off")
plt.show()

### The effect of temperature

Sweeping the temperature makes the diversity/typicality trade-off visible directly.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 6))
for ax, temp in zip(axs, [0.5, 1.0, 1.5]):
    with torch.no_grad():
        s = prior.sample(num_samples=9, temperature=temp)
    grid = make_grid(s.cpu(), nrow=3, normalize=True, padding=2)
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(f"temperature = {temp}", fontsize=15)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

- A VQ-VAE encodes an image to a **grid of discrete codebook indices** rather than a continuous Gaussian latent.
- Because of that, a trained VQ-VAE can *reconstruct* but **cannot generate** — it has no prior over index grids.
- `predict_step` still yields uncertainty, obtained from stochastic codebook sampling, which requires `sample_codebook_temp > 0` and `num_samples > 1`.
- **Watch codebook perplexity and usage, not just the loss.** A low reconstruction loss with a collapsed codebook is a failed model.
- Generation comes from a second stage: `VQVAEPrior` trains a PixelCNN over the index grid, and `sample()` draws from it ancestrally, one grid position at a time.

The same two-stage recipe scales to larger images by increasing the encoder depth or input size — but remember that sampling cost grows with the *square* of the grid side, so a 16x16 grid is four times more sequential passes than an 8x8 one.